# DyGEnc smoke test trên Kaggle T4

Bật **Internet**, chọn accelerator **T4**, Add Input dataset `tdat1465/agqa-balanced`, rồi tạo Kaggle Secret `HF_TOKEN` có quyền đọc gated model `meta-llama/Llama-3.2-3B`. Bản này chỉ test 8 video/128 QA mỗi split và 2 optimizer updates; không phải phép đo accuracy của paper.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/tdat1465/DyGEnc.git'
BRANCH = 'codex/kaggle-t4'
REPO_DIR = Path('/kaggle/working/DyGEnc-kaggle-t4')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch',
                    BRANCH, REPO_URL, str(REPO_DIR)], env=git_env, check=True)
else:
    assert (REPO_DIR / '.git').is_dir(), 'REPO_DIR tồn tại nhưng không phải Git checkout'
    status = subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--porcelain'],
                            env=git_env, check=True, text=True, capture_output=True).stdout
    assert status == '', 'Repo có thay đổi chưa lưu; dùng thư mục clone khác'
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1',
                    'origin', f'refs/heads/{BRANCH}'], env=git_env, check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'FETCH_HEAD'],
                   env=git_env, check=True)

commit = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], env=git_env,
                        check=True, text=True, capture_output=True).stdout.strip()
print('Code:', REPO_DIR)
print('Commit:', commit)

## Chạy probe

Launcher tạo venv riêng nhưng dùng lại Torch 2.10/cu128 của Kaggle, tải model vào working storage, đọc trực tiếp bốn raw file (không copy Charades), preprocess subset rồi train bằng FP16 + gradient scaling. Secret chỉ được đưa vào environment của tiến trình con và bị xóa khỏi namespace ngay sau đó.

In [ ]:
from kaggle_secrets import UserSecretsClient

mount_candidates = [
    Path('/kaggle/input/datasets/tdat1465/agqa-balanced'),
    Path('/kaggle/input/agqa-balanced'),
]
MOUNTED_ROOT = next((path for path in mount_candidates if path.is_dir()), None)
assert MOUNTED_ROOT is not None, 'Hãy Add Input dataset tdat1465/agqa-balanced'
WORK_ROOT = Path('/kaggle/working/dygenc-t4-work')
RUN_DIR = Path('/kaggle/working/dygenc-t4-run')

def run_t4(mode, stop_after_updates):
    command = [
        'python', str(REPO_DIR / 'scripts/kaggle/run_agqa_t4.py'),
        '--mounted-root', str(MOUNTED_ROOT),
        '--work-root', str(WORK_ROOT),
        '--run-dir', str(RUN_DIR),
        '--gpu-index', '0',
        '--mode', mode,
        '--smoke-videos-per-split', '8',
        '--smoke-qa-per-split', '128',
        '--accumulation-steps', '32',
        '--stop-after-updates', str(stop_after_updates),
        '--checkpoint-every', '1',
    ]
    child_env = os.environ.copy()
    token = UserSecretsClient().get_secret('HF_TOKEN')
    child_env['HF_TOKEN'] = token
    process = subprocess.Popen(command, env=child_env)
    try:
        returncode = process.wait()
    except KeyboardInterrupt:
        # Yêu cầu launcher chuyển SIGTERM cho trainer và đợi checkpoint an toàn.
        process.terminate()
        returncode = process.wait()
    finally:
        child_env.pop('HF_TOKEN', None)
        del token, child_env
    if returncode not in (0, 75):
        raise RuntimeError(f'Launcher failed with exit code {returncode}')
    return returncode

returncode = run_t4('fresh', 2)
if returncode == 75:
    print('Probe hoàn tất đúng dự kiến: checkpoint đã lưu trước exit 75.')
assert (RUN_DIR / 'last.pth').is_file()
print('Checkpoint:', RUN_DIR / 'last.pth')

## Resume trong cùng session (tùy chọn)

Cell này chạy hết tập smoke qua tổng cộng 5 epoch. Giữ nguyên mọi cấu hình; `stop_after_updates=0` không biến nó thành full-data training.

In [ ]:
assert (RUN_DIR / 'last.pth').is_file(), 'Chưa có checkpoint để resume'
returncode = run_t4('resume', 0)
print('Resume exit:', returncode)

## Giữ checkpoint qua session

Dùng **Save Version** để lưu `/kaggle/working/dygenc-t4-run`. Ở session mới, Add Input output đó và copy `last.pth`, `model-revisions.json`, `raw-data-manifest.json` (cùng `best.pth` nếu có) vào `RUN_DIR`, rồi chạy cell resume. Chỉ sử dụng checkpoint do chính bạn tạo vì định dạng PyTorch này không an toàn từ nguồn lạ. Xem hướng dẫn đầy đủ trong `KAGGLE_T4.md`.